# Questão 6 — Dimensão de Calendário

## Objetivo
Identificar qual dia da semana apresenta a menor média histórica de vendas, considerando corretamente os dias em que a loja esteve aberta, mas não houve nenhuma venda registrada.

## Contexto de negócio
A análise não pode ser feita apenas com a tabela de vendas, porque dias sem transações não aparecem no dataset transacional. Se esses dias forem ignorados, a média de vendas por dia da semana fica artificialmente inflada.

## Abordagem técnica
Para resolver o problema, será utilizada uma dimensão de datas (`dim_data`) já construída no dbt e armazenada na camada gold, juntamente com a tabela fato `fct_vendas`. A análise seguirá estas etapas:

1. Validar o intervalo de datas analisado.
2. Agregar as vendas por dia.
3. Cruzar o calendário completo com a fato de vendas via `LEFT JOIN`.
4. Substituir dias sem venda por zero.
5. Calcular a média histórica por dia da semana.
6. Identificar o pior dia da semana em termos de média de vendas.

## Tabelas utilizadas
- `lh_nautical.04_gold.dim_data`
- `lh_nautical.04_gold.fct_vendas`

## Observação sobre a modelagem

A dimensão de datas já foi criada anteriormente no dbt como parte do modelo dimensional do projeto. Segundo o contexto técnico do projeto, a `dim_data` possui uma linha por dia e inclui todas as datas do período de vendas, inclusive dias sem movimento, exatamente para suportar análises como esta.

Mesmo assim, neste notebook será demonstrada também a lógica SQL equivalente de construção do calendário, para deixar explícito o raciocínio pedido na questão.

In [0]:
%sql
-- Validar o intervalo de datas existente na fato de vendas.
-- Esta etapa confirma que o período analisado será exatamente
-- da menor até a maior data presente nas vendas, conforme a premissa da questão.

select
    min(sale_date) as data_minima_vendas,
    max(sale_date) as data_maxima_vendas,
    count(*) as total_registros_vendas
from lh_nautical.04_gold.fct_vendas;

data_minima_vendas,data_maxima_vendas,total_registros_vendas
2023-01-01,2024-12-31,9895


In [0]:
%sql
-- Conferir se a dimensão de datas cobre corretamente o intervalo completo.
-- Como a tabela possui uma linha por dia, ela deve refletir todo o período analisado.

select
    min(data_dia) as data_minima_dim,
    max(data_dia) as data_maxima_dim,
    count(*) as total_dias_calendario
from lh_nautical.04_gold.dim_data;

data_minima_dim,data_maxima_dim,total_dias_calendario
2023-01-01,2024-12-31,731


## Construção da análise

A lógica abaixo faz o seguinte:

- agrega as vendas da fato em nível diário;
- cruza esse resultado com a dimensão de datas por meio de `LEFT JOIN`;
- preenche com zero os dias sem venda;
- calcula a média histórica de vendas por dia da semana.

Esse desenho garante que todos os dias do calendário sejam considerados no denominador da média.

In [0]:
%sql
-- Questão 6.1
-- Cálculo da média de vendas por dia da semana considerando dias sem venda.
-- utilizada a dimensão de datas já construída no dbt.

with vendas_diarias as (

    -- Agrega a fato para obter a venda total por dia.
    -- Como a granularidade da fato é 1 linha por venda, é necessário consolidar o valor diário.
    select
        sale_date,
        sum(receita_transacao_brl) as valor_venda_dia
    from lh_nautical.04_gold.fct_vendas
    group by sale_date

),

calendario_com_vendas as (

    -- LEFT JOIN para preservar todas as datas do calendário.
    -- Quando não houver venda em determinada data, o valor será tratado como zero.
    select
        d.data_dia,
        d.dia_semana,
        d.numero_dia_semana,
        coalesce(v.valor_venda_dia, 0) as valor_venda_dia
    from lh_nautical.04_gold.dim_data d
    left join vendas_diarias v
        on d.data_dia = v.sale_date

),

media_por_dia_semana as (

    -- Calcula a média histórica da venda diária por dia da semana,
    -- agora considerando também os dias zerados.
    select
        dia_semana,
        numero_dia_semana,
        round(avg(valor_venda_dia), 2) as media_venda_diaria
    from calendario_com_vendas
    group by dia_semana, numero_dia_semana

)

select
    dia_semana,
    media_venda_diaria
from media_por_dia_semana
order by media_venda_diaria asc, numero_dia_semana asc;

dia_semana,media_venda_diaria
Domingo,3319503.57
Segunda-feira,3465137.71
Quarta-feira,3535265.63
Quinta-feira,3626232.44
Terça-feira,3627045.76
Sábado,3710540.55
Sexta-feira,3715003.41


In [0]:
%sql
-- Versão alternativa da Questão 6.1
-- Esta consulta demonstra explicitamente a construção do calendário em SQL,
-- sem depender da dimensão já materializada.

/*with limites as (

    select
        min(sale_date) as data_minima,
        max(sale_date) as data_maxima
    from lh_nautical.04_gold.fct_vendas

),

calendario as (

    -- Gera uma linha por dia entre a data mínima e a data máxima das vendas.
    select
        explode(
            sequence(
                (select data_minima from limites),
                (select data_maxima from limites),
                interval 1 day
            )
        ) as data_dia

),

dim_calendario as (

    -- Enriquecimento do calendário com atributos de data e nome do dia da semana em português.
    select
        data_dia,
        year(data_dia) as ano,
        quarter(data_dia) as trimestre,
        month(data_dia) as mes,
        day(data_dia) as dia,
        dayofweek(data_dia) as numero_dia_semana,
        case dayofweek(data_dia)
            when 1 then 'Domingo'
            when 2 then 'Segunda-feira'
            when 3 then 'Terça-feira'
            when 4 then 'Quarta-feira'
            when 5 then 'Quinta-feira'
            when 6 then 'Sexta-feira'
            when 7 then 'Sábado'
        end as dia_semana
    from calendario

),

vendas_diarias as (

    -- Soma da venda por data.
    select
        sale_date,
        sum(receita_transacao_brl) as valor_venda_dia
    from lh_nautical.04_gold.fct_vendas
    group by sale_date

),

calendario_com_vendas as (

    -- Preserva todo o calendário e trata ausência de venda como zero.
    select
        c.data_dia,
        c.dia_semana,
        c.numero_dia_semana,
        coalesce(v.valor_venda_dia, 0) as valor_venda_dia
    from dim_calendario c
    left join vendas_diarias v
        on c.data_dia = v.sale_date

)

select
    dia_semana,
    round(avg(valor_venda_dia), 2) as media_venda_diaria
from calendario_com_vendas
group by dia_semana, numero_dia_semana
order by media_venda_diaria asc, numero_dia_semana asc;

## Validação da resposta

A consulta abaixo retorna apenas o dia da semana com a menor média histórica de vendas, já considerando os dias sem qualquer venda registrada.

In [0]:
%sql
-- Questão 6.2
-- Retorna somente o pior dia da semana e sua média histórica de vendas.

with vendas_diarias as (

    select
        sale_date,
        sum(receita_transacao_brl) as valor_venda_dia
    from lh_nautical.04_gold.fct_vendas
    group by sale_date

),

calendario_com_vendas as (

    select
        d.data_dia,
        d.dia_semana,
        d.numero_dia_semana,
        coalesce(v.valor_venda_dia, 0) as valor_venda_dia
    from lh_nautical.04_gold.dim_data d
    left join vendas_diarias v
        on d.data_dia = v.sale_date

),

media_por_dia_semana as (

    select
        dia_semana,
        numero_dia_semana,
        round(avg(valor_venda_dia), 2) as media_venda_diaria
    from calendario_com_vendas
    group by dia_semana, numero_dia_semana

)

select
    dia_semana,
    media_venda_diaria
from media_por_dia_semana
order by media_venda_diaria asc, numero_dia_semana asc
limit 1;

dia_semana,media_venda_diaria
Domingo,3319503.57


In [0]:
%sql
-- Consulta para evidenciar que existem dias sem venda.


with vendas_diarias as (

    select
        sale_date,
        sum(receita_transacao_brl) as valor_venda_dia
    from lh_nautical.04_gold.fct_vendas
    group by sale_date

)

select
    d.data_dia,
    d.dia_semana,
    coalesce(v.valor_venda_dia, 0) as valor_venda_dia
from lh_nautical.04_gold.dim_data d
left join vendas_diarias v
    on d.data_dia = v.sale_date
where coalesce(v.valor_venda_dia, 0) = 0
order by d.data_dia;

data_dia,dia_semana,valor_venda_dia
2023-01-15,Domingo,0.00
2023-03-21,Terça-feira,0.00
2023-07-30,Domingo,0.00
2023-08-14,Segunda-feira,0.00
2024-05-14,Terça-feira,0.00
2024-12-09,Segunda-feira,0.00


## Questão 6.3 — Explicação

É necessário utilizar uma tabela de datas porque a tabela transacional de vendas registra apenas os dias em que houve alguma venda. Portanto, se a análise for feita diretamente sobre ela, os dias em que a loja esteve aberta mas vendeu zero simplesmente desaparecem do cálculo.

Ao usar uma dimensão de calendário, todas as datas do período passam a existir explicitamente na análise. Com isso, o `LEFT JOIN` garante que datas sem venda também sejam consideradas, e o `COALESCE` transforma a ausência de registro em valor zero.

Se um determinado dia da semana tiver muitos dias sem nenhuma venda registrada e esses dias forem ignorados, a média desse dia ficará artificialmente maior do que a realidade. Em outras palavras, a análise passa a refletir apenas os “dias bons” daquele dia da semana, produzindo um viés otimista e podendo induzir a diretoria a uma decisão incorreta.

## Após considerar corretamente os dias sem venda no cálculo, o dia da semana com a menor média histórica de vendas foi Domingo, com média de R$ 3.319.503,57.